# The Bottleneck-Within-the-Bottleneck — power-equipment valuation screen

Thesis ([[power-scarcity-equities]], [[buildout-bottleneck-map]]): power is the constraint on the AI
buildout, but the *obvious* power plays — the independent power producers (IPPs: CEG/VST/TLN) — have
**already run** on that consensus. The tighter, less-crowded constraint is the layer that actually
connects the gigawatts: **grid / transformers / switchgear / electrical equipment / power-construction.**
You can't energize a data center without transformers and electricians, and both are in shortage
(the Beige Book flagged both today).

**What this does:** for the equipment layer, it measures *how much each name has already run* (52-week
position, 1yr return) against *how it's valued* (forward P/E, EV/EBITDA, FCF yield) — so you can spot
the ones the market hasn't fully bid up yet. The **IPP** and **regulated-utility** groups are included
as reference: IPPs = the 'already priced' benchmark, utilities = the 'cheap but regulated/slow' benchmark.

---
### Honest limits (read once)
- **yfinance `.info` fundamentals are TTM, point-in-time, and flaky** — some cells will come back blank.
  Treat every multiple as *directional*, not precise. Price-based columns (returns, 52w position) are the
  reliable ones.
- **Cheap != good.** A low multiple can mean value-trap, not bargain. The FCF-yield column separates
  'cheap with cash' from 'cheap and bleeding.' This screen ranks *relative valuation*, not quality.
- **US-listed proxies only.** The actual transformer kings (ABB, Siemens Energy, Hitachi Energy) are
  foreign; this captures the US electrical-equipment layer, not pure transformers.
- **Candidate generator, not a buy list.** It surfaces names for deeper work. Sizing/quality is yours.


In [ ]:
# !pip install yfinance --quiet
import numpy as np, pandas as pd, yfinance as yf
import warnings; warnings.filterwarnings('ignore')
pd.set_option('display.width', 220); pd.set_option('display.max_columns', 40)


## CONFIG — the universe, grouped


In [ ]:
GROUPS = {
  'EQUIP/GRID': ['ETN','GEV','VRT','HUBB','POWL','NVT','ATKR','PWR','MYRG','MTZ','ENS','PNR','ROK','EMR','GNRC','AYI'],
  'IPP (ran)' : ['CEG','VST','TLN','NRG'],           # already-priced reference
  'UTIL (reg)': ['NEE','SO','DUK','AEP','D'],         # cheap-but-slow reference
  'OTHER'     : ['BE','CCJ'],                          # distributed / uranium
}
PERF_LOOKBACK = '1y'


## Fetch: price behavior (reliable) + fundamentals (best-effort)


In [ ]:
ALL = [t for g in GROUPS.values() for t in g]
bucket = {t:g for g,ts in GROUPS.items() for t in ts}
print(f'Pulling {len(ALL)} tickers ...')
px = yf.download(ALL, period=PERF_LOOKBACK, auto_adjust=True, progress=False, group_by='column')
close = px['Close'] if isinstance(px.columns, pd.MultiIndex) else px[['Close']]
high  = px['High']  if isinstance(px.columns, pd.MultiIndex) else px[['High']]
low   = px['Low']   if isinstance(px.columns, pd.MultiIndex) else px[['Low']]

INFO_KEYS = ['marketCap','trailingPE','forwardPE','enterpriseToEbitda',
             'priceToSalesTrailing12Months','freeCashflow','revenueGrowth']
def safe_info(t):
    try: info = yf.Ticker(t).info
    except Exception: info = {}
    return {k: info.get(k, np.nan) for k in INFO_KEYS}

rows=[]
for t in ALL:
    try:
        c = close[t].dropna()
        if len(c) < 60: continue
        last = float(c.iloc[-1]); lo = float(low[t].min()); hi = float(high[t].max())
        pos_52w = (last-lo)/(hi-lo) if hi>lo else np.nan   # 0=at lows, 1=at highs
        ret_1y  = (last/float(c.iloc[0])-1)*100
        ret_6m  = (last/float(c.iloc[max(0,len(c)-126)])-1)*100
        info = safe_info(t)
        mc = info['marketCap']; fcf = info['freeCashflow']
        fcf_yield = (fcf/mc*100) if (mc and fcf and mc==mc and fcf==fcf) else np.nan
        rows.append(dict(ticker=t, group=bucket[t], price=round(last,2),
            pos_52w=round(pos_52w,2), ret_1y=round(ret_1y,1), ret_6m=round(ret_6m,1),
            fwdPE=info['forwardPE'], EV_EBITDA=info['enterpriseToEbitda'],
            PS=info['priceToSalesTrailing12Months'], FCF_yld=round(fcf_yield,2) if fcf_yield==fcf_yield else np.nan,
            rev_growth=round(info['revenueGrowth']*100,1) if info['revenueGrowth']==info['revenueGrowth'] else np.nan,
            mktcap_B=round(mc/1e9,1) if (mc and mc==mc) else np.nan))
    except Exception as e:
        print('skip', t, e)
df = pd.DataFrame(rows).set_index('ticker')
print('Got', len(df), 'names')


## Rank: which equipment names are still cheap vs already bid up


In [ ]:
eq = df[df['group']=='EQUIP/GRID'].copy()

# 'already run' score: high 52w position + high 1yr return = the market already loves it
def z(s): return (s - s.mean())/s.std(ddof=0) if s.std(ddof=0)>0 else s*0
eq['run_score']   = (z(eq['pos_52w'].fillna(eq['pos_52w'].median())) + z(eq['ret_1y'].fillna(eq['ret_1y'].median())))/2
# 'value' score: cheap on EV/EBITDA + high FCF yield (higher value_score = cheaper)
eq['value_score'] = (-z(eq['EV_EBITDA'].fillna(eq['EV_EBITDA'].median())) + z(eq['FCF_yld'].fillna(eq['FCF_yld'].median())))/2
# 'still-cheap-in-the-bottleneck': cheap AND not yet fully run
eq['gap_score']   = eq['value_score'] - eq['run_score']

cols = ['group','price','pos_52w','ret_1y','ret_6m','fwdPE','EV_EBITDA','FCF_yld','rev_growth','mktcap_B','gap_score']
print('=== EQUIPMENT/GRID LAYER  (sorted: most cheap-and-un-run at TOP) ===')
print(eq.sort_values('gap_score', ascending=False)[cols].round(2).to_string())

print('\n=== REFERENCE GROUPS (context) ===')
ref = df[df['group']!='EQUIP/GRID']
print(ref.sort_values(['group','pos_52w'])[['group','price','pos_52w','ret_1y','fwdPE','EV_EBITDA','FCF_yld','mktcap_B']].round(2).to_string())


### How to read it
- **pos_52w** — where it sits in its 1-yr range. **Near 1.0 = already run** (priced in); **lower = the
  market hasn't chased it yet.** This is the most reliable 'already priced?' read (pure price, no `.info`).
- **ret_1y / ret_6m** — corroborates: big numbers = already moved.
- **EV_EBITDA / fwdPE** — valuation; lower = cheaper (but check rev_growth — cheap + shrinking = trap).
- **FCF_yld** — free-cash-flow yield; higher = more cash for the price. Separates real value from junk.
- **gap_score** — the composite: high = *cheap AND not yet run* = the un-crowded corner of the bottleneck.
- **Compare to the reference rows:** if the IPPs (CEG/VST/TLN) are pinned near pos_52w ~1.0 with rich
  multiples, that's the 'already priced' the thesis warned about — and any EQUIP name sitting at pos_52w
  ~0.5 with a real FCF yield is the 'same bottleneck, not yet bid up' candidate.

**Next step after this:** take the top 2-3 gap_score names and run them through the mean-reversion /
behavior screener, and read their actual FCF/capex trend — this screen only tells you *relatively cheap*,
not *good*. It's the candidate generator; the quality read is the second pass.
